In [3]:
import pandas as pd
import numpy as np
from datetime import datetime
import calendar



disasters = pd.read_csv("C:/Users/zhoum/Documents/company_disaster_post2014.csv")
recovery  = pd.read_csv("C:/Users/zhoum/Downloads/storm_company_recovery_days.csv")


disaster_recovery = disasters.merge(recovery, on=["ticker", "event_name"], how="left")


disaster_recovery = disaster_recovery[disaster_recovery['disaster_year'] >= 2015]


disaster_recovery['start_date'] = pd.to_datetime(disaster_recovery['start_date'], format="%Y/%m/%d")
disaster_recovery['end_date']   = pd.to_datetime(disaster_recovery['end_date'], format="%Y/%m/%d")


disaster_recovery['start_month'] = disaster_recovery['start_date'].dt.month
disaster_recovery['end_month']   = disaster_recovery['end_date'].dt.month

disaster_recovery['duration'] = (disaster_recovery['end_date'] - disaster_recovery['start_date']).dt.days

cols = list(disaster_recovery.columns)
insert_at = cols.index('disaster_year') + 1
for col in ['start_month', 'end_month', 'duration']:
    cols.insert(insert_at, cols.pop(cols.index(col)))
disaster_recovery = disaster_recovery[cols]


disaster_recovery['rec_1d']   = ((disaster_recovery['recovery_days'] != -1) & (disaster_recovery['recovery_days'] <= 1)).astype(int)
disaster_recovery['rec_7d']   = ((disaster_recovery['recovery_days'] != -1) & (disaster_recovery['recovery_days'] <= 7)).astype(int)
disaster_recovery['rec_30d']  = ((disaster_recovery['recovery_days'] != -1) & (disaster_recovery['recovery_days'] <= 30)).astype(int)
disaster_recovery['rec_180d'] = ((disaster_recovery['recovery_days'] != -1) & (disaster_recovery['recovery_days'] <= 180)).astype(int)







In [4]:
disaster_recovery

,ticker,company_name,event_name,disaster_subtype,magnitude_kph,total_damage_adjusted,start_date,end_date,disaster_year,duration,...,pre_shock_price,recovery_date,recovery_price,recovery_days,benchmark_available,recovered_by_end,rec_1d,rec_7d,rec_30d,rec_180d
11,A,"AGILENT TECHNOLOGIES, INC.",2015_apr_storm1,Tornado,NaN,1852882.0,2015-04-07,2015-04-10,2015,3,...,41.930000,2015-04-07 00:00:00,42.439999,0.0,1.0,1.0,1,1,1,1
12,A,"AGILENT TECHNOLOGIES, INC.",2015_apr_storm2,Tornado,NaN,1257313.0,2015-04-24,2015-04-28,2015,4,...,42.619999,2015-05-11 00:00:00,42.619999,11.0,1.0,1.0,0,0,1,1
13,A,"AGILENT TECHNOLOGIES, INC.",2015_aug_storm1,Severe weather,95.0,1257313.0,2015-08-02,2015-08-04,2015,2,...,40.950001,2015-08-03 00:00:00,41.000000,0.0,1.0,1.0,1,1,1,1
14,A,"AGILENT TECHNOLOGIES, INC.",2015_dec_storm1,Tornado,NaN,2646974.0,2015-12-26,2015-12-30,2015,4,...,42.139999,2015-12-29 00:00:00,42.360001,1.0,1.0,1.0,1,1,1,1
15,A,"AGILENT TECHNOLOGIES, INC.",2015_feb_storm1,Blizzard/Winter storm,NaN,3970461.0,2015-02-16,2015-02-22,2015,6,...,40.150002,2015-02-17 00:00:00,40.520000,0.0,1.0,1.0,1,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58049,ZTS,ZOETIS INC.,2022_may_storm2,Derecho,NaN,2679683.0,2022-05-19,2022-05-26,2022,7,...,158.860001,2022-05-20 00:00:00,162.559998,1.0,1.0,1.0,1,1,1,1
58050,ZTS,ZOETIS INC.,2023_apr_storm1,Tornado,NaN,2985536.0,2023-04-03,2023-04-07,2023,4,...,166.440002,2023-04-03 00:00:00,167.660004,0.0,1.0,1.0,1,1,1,1
58051,ZTS,ZOETIS INC.,2023_jun_storm2,Storm (General),NaN,3912081.0,2023-06-15,2023-06-22,2023,7,...,165.649994,2023-06-15 00:00:00,171.429993,0.0,1.0,1.0,1,1,1,1
58052,ZTS,ZOETIS INC.,2024_apr_storm1,Tornado,NaN,1725000.0,2024-04-25,2024-05-08,2024,13,...,150.880005,2024-04-25 00:00:00,153.360001,0.0,1.0,1.0,1,1,1,1


In [8]:

real_GDP = pd.read_csv("C:/Users/zhoum/Downloads/GDPCA.csv")


real_GDP['year'] = pd.to_datetime(real_GDP['observation_date'], format="%d/%m/%Y").dt.year


real_GDP = real_GDP[['year', 'GDPCA']]

disaster_recovery = disaster_recovery.merge(real_GDP, left_on='disaster_year', right_on='year', how='left')
disaster_recovery.drop(columns=['year'], inplace=True)

inflation = pd.read_csv("C:/Users/zhoum/Downloads/inflation.csv")
inflation['year'] = pd.to_datetime(inflation['observation_date']).dt.year
inflation = inflation[['year', 'FPCPITOTLZGUSA']].rename(columns={'FPCPITOTLZGUSA': 'inflation'})

disaster_recovery = disaster_recovery.merge(inflation, left_on='disaster_year', right_on='year', how='left')
disaster_recovery.drop(columns=['year'], inplace=True)

In [9]:

northeast = ["Maine", "New Hampshire", "Vermont", "Massachusetts", 
             "Rhode Island", "Connecticut", "New York", "New Jersey", "Pennsylvania"]

midwest = ["Ohio", "Indiana", "Illinois", "Michigan", "Wisconsin",
           "Minnesota", "Iowa", "Missouri", "North Dakota", 
           "South Dakota", "Nebraska", "Kansas"]

south = ["Delaware", "Maryland", "District of Columbia", "Virginia", "West Virginia",
         "North Carolina", "South Carolina", "Georgia", "Florida",
         "Kentucky", "Tennessee", "Mississippi", "Alabama", "Oklahoma", "Texas", "Arkansas", "Louisiana"]

west = ["Montana", "Idaho", "Wyoming", "Colorado", "New Mexico", "Arizona", "Utah", "Nevada",
        "Washington", "Oregon", "California", "Alaska", "Hawaii"]

def get_region(state):
    if state in northeast:
        return "Northeast"
    elif state in midwest:
        return "Midwest"
    elif state in south:
        return "South"
    elif state in west:
        return "West"
    else:
        return "Other"

disaster_recovery['region'] = disaster_recovery['affected_states'].apply(get_region)


cols = list(disaster_recovery.columns)
cols.insert(cols.index('affected_states') + 1, cols.pop(cols.index('region')))
disaster_recovery = disaster_recovery[cols]


disaster_recovery = disaster_recovery[~disaster_recovery['pre_shock_price'].isna()]
disaster_recovery = disaster_recovery.drop(columns=['Investments', 'magnitude_kph'])

na_percent = disaster_recovery.isna().mean() * 100
print(na_percent)

disaster_recovery.to_csv("predictor_disaster_data_python.csv", index=False)
print("Saved to predictor_disaster_data.csv")



ticker                                  0.000000
company_name                            0.000000
event_name                              0.000000
disaster_subtype                        0.000000
total_damage_adjusted                   0.000000
start_date                              0.000000
end_date                                0.000000
disaster_year                           0.000000
duration                                0.000000
end_month                               0.000000
start_month                             0.000000
affected_states                         0.000000
region                                  0.000000
upstream                                0.568569
downstream                              0.568569
assets                                  1.449852
assets_current                         17.762111
cash_and_equivalents                   13.880676
stockholders_equity                     9.199454
revenue                                22.251914
operating_income_los

In [10]:
import os

# Get current working directory
cwd = os.getcwd()
print("Current working directory:", cwd)


Current working directory: C:\Users\zhoum
